# LightGBM Model

In [11]:
# =============================================================================
# LightGBM MODEL: CHIRPS + ERA5 (inputs) -> CDI (target)
# TRAINING, VALIDATION, AND TESTING PERFORMANCE WITH 3-WAY SPLIT
# =============================================================================

import os
import glob
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import rasterio
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import lightgbm as lgb

# =============================================================================
# SETTINGS
# =============================================================================

SEED       = 42
TRAIN_END  = 2017       # Training: 2000-2017
VAL_END    = 2021       # Validation: 2018-2021 (used for early stopping)
# Testing    = 2022-2025   # Testing: 2022-2025 (completely unseen holdout)

CDI_PREFIX = "CDI_"

BASE_DIR   = r"C:\Users\gerii\Documents\projects\tekeze"
CHIRPS_DIR = os.path.join(BASE_DIR, "CHIRPS_resampled")
ERA5_DIR   = os.path.join(BASE_DIR, "ERA5_resampled")
CDI_DIR    = os.path.join(BASE_DIR, "indices", "CDI_OUTPUT_025")
OUT_DIR    = os.path.join(BASE_DIR, "lgbm_results_final")
os.makedirs(OUT_DIR, exist_ok=True)

FEATURES = ["precip", "tmean", "precip_lag1", "tmean_lag1",
            "precip_3m", "precip_6m", "tmean_3m"]

FS = 23
plt.rcParams.update({
    "font.size": FS, "axes.labelsize": FS, "axes.titlesize": FS,
    "xtick.labelsize": FS - 2, "ytick.labelsize": FS - 2,
    "legend.fontsize": FS - 4, "font.family": "serif", "axes.linewidth": 1.2,
})

params = {
    "objective": "regression", "metric": "rmse", "boosting_type": "gbdt",
    "num_leaves": 127, "max_depth": -1, "learning_rate": 0.05,
    "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 1.0,
    "reg_alpha": 0.01, "reg_lambda": 1.0, "min_child_samples": 20,
    "verbose": -1, "seed": SEED, "n_jobs": -1,
}

# =============================================================================
# LOAD DATA
# =============================================================================

def get_keys(folder, prefix=""):
    keys = set()
    for fp in glob.glob(os.path.join(folder, f"{prefix}*.tif")):
        name = os.path.basename(fp).replace(".tif", "")
        if prefix and name.startswith(prefix):
            name = name[len(prefix):]
        keys.add(name)
    return keys

def load_band(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        nd = src.nodata
    if nd is not None and not np.isnan(nd):
        arr[arr == nd] = np.nan
    arr[arr <= -9999] = np.nan
    return arr

def back(year, month, k):
    idx = year * 12 + (month - 1) - k
    return f"{idx // 12}_{idx % 12 + 1:02d}"

print("=" * 70)
print("LightGBM: CHIRPS + ERA5 -> CDI | Train, Validation, and Test Split")
print("=" * 70)

climate_keys = get_keys(CHIRPS_DIR) & get_keys(ERA5_DIR)
common_keys  = sorted(climate_keys & get_keys(CDI_DIR, CDI_PREFIX))
if not common_keys:
    raise SystemExit("No common months found. Check your folder paths.")
print(f"CDI months: {len(common_keys)} ({common_keys[0]} to {common_keys[-1]})")

P = {k: load_band(os.path.join(CHIRPS_DIR, f"{k}.tif")) for k in sorted(climate_keys)}
T = {k: load_band(os.path.join(ERA5_DIR,   f"{k}.tif")) for k in sorted(climate_keys)}

cols = {c: [] for c in FEATURES + ["month", "year", "row", "col", "cdi"]}
for key in common_keys:
    y_, m_ = int(key[:4]), int(key[5:7])
    lk = [back(y_, m_, k) for k in range(6)]
    if not all(k in P for k in lk):
        continue
    c = load_band(os.path.join(CDI_DIR, f"{CDI_PREFIX}{key}.tif"))
    if not (P[lk[0]].shape == T[lk[0]].shape == c.shape):
        continue
    f = {
        "precip": P[lk[0]], "tmean": T[lk[0]],
        "precip_lag1": P[lk[1]], "tmean_lag1": T[lk[1]],
        "precip_3m": np.sum([P[k] for k in lk[:3]], axis=0),
        "precip_6m": np.sum([P[k] for k in lk[:6]], axis=0),
        "tmean_3m":  np.mean([T[k] for k in lk[:3]], axis=0),
    }
    valid = np.isfinite(c)
    for a in f.values():
        valid &= np.isfinite(a)
    rr, cc = np.nonzero(valid)
    n = len(rr)
    for name, a in f.items():
        cols[name].append(a[valid])
    cols["month"].append(np.full(n, m_, dtype=np.float32))
    cols["year"].append(np.full(n, y_, dtype=np.int16))
    cols["row"].append(rr.astype(np.int32))
    cols["col"].append(cc.astype(np.int32))
    cols["cdi"].append(c[valid])
del P, T

df = pd.DataFrame({k: np.concatenate(v) for k, v in cols.items()})
print(f"Total samples across all years: {len(df):,}")
yr, y_all = df["year"].values, df["cdi"].values

# Define masks for the 3 sets
masks = {
    "Training": yr <= TRAIN_END,
    "Validation": (yr > TRAIN_END) & (yr <= VAL_END),
    "Testing": yr > VAL_END
}
SETS = list(masks)

# =============================================================================
# TRAIN WITH EARLY STOPPING
# =============================================================================

X = df[FEATURES].values
scaler = StandardScaler()

# Fit scaler ONLY on training data, then transform all
Xs = {
    "Training": scaler.fit_transform(X[masks["Training"]]),
    "Validation": scaler.transform(X[masks["Validation"]]),
    "Testing": scaler.transform(X[masks["Testing"]])
}

d_tr = lgb.Dataset(Xs["Training"], label=y_all[masks["Training"]],
                   feature_name=FEATURES, free_raw_data=False)
d_va = lgb.Dataset(Xs["Validation"], label=y_all[masks["Validation"]],
                   reference=d_tr, free_raw_data=False)
d_te = lgb.Dataset(Xs["Testing"], label=y_all[masks["Testing"]],
                   reference=d_tr, free_raw_data=False)

evals = {}
# Early stopping uses the validation set to find the best stopping round
model = lgb.train(
    params, d_tr,
    num_boost_round=1500,
    valid_sets=[d_tr, d_va],
    valid_names=["train", "val"],
    callbacks=[
        lgb.log_evaluation(50),
        lgb.early_stopping(stopping_rounds=50),
        lgb.record_evaluation(evals)
    ]
)

best_round = model.best_iteration
print(f"\nBest boosting round selected by validation set: {best_round}")

# Save final artifacts
model.save_model(os.path.join(OUT_DIR, "lgbm_model_final.txt"))
joblib.dump(scaler, os.path.join(OUT_DIR, "scaler_final.joblib"))
with open(os.path.join(OUT_DIR, "final_features.json"), "w") as f:
    json.dump({"features": FEATURES, "best_rounds": best_round,
               "train_end": TRAIN_END, "val_end": VAL_END}, f, indent=2)

preds = {k: model.predict(Xs[k]) for k in SETS}

# =============================================================================
# TABLES
# =============================================================================

def stats(o, p):
    return {
        "R2": r2_score(o, p), 
        "r": float(np.corrcoef(o, p)[0, 1]),
        "RMSE": float(np.sqrt(mean_squared_error(o, p))),
        "MAE": mean_absolute_error(o, p), 
        "ME": float(np.mean(p - o))
    }

t1, t2, met = [], [], {}
for k in SETS:
    m = masks[k]
    o = y_all[m]
    t1.append([
        k, f"{yr[m].min()}-{yr[m].max()}",
        df.loc[m, ["year", "month"]].drop_duplicates().shape[0],
        df.loc[m, ["row", "col"]].drop_duplicates().shape[0],
        int(m.sum()), round(100 * m.sum() / len(df), 1),
        round(o.mean(), 2), round(o.std(), 2),
        round(o.min(), 2), round(o.max(), 2)
    ])
    met[k] = stats(o, preds[k])
    t2.append([k, int(m.sum())] + [round(met[k][c], 3) for c in ["R2", "r", "RMSE", "MAE", "ME"]])

T1 = pd.DataFrame(t1, columns=["Dataset", "Years", "Months", "Pixels", "Samples",
                               "Share (%)", "CDI mean", "CDI SD", "CDI min", "CDI max"])
T2 = pd.DataFrame(t2, columns=["Dataset", "Samples", "R2", "r", "RMSE", "MAE", "Mean error"])
T3 = pd.DataFrame([[k, v] for k, v in {**params, "best_rounds": best_round,
                                     "inputs": ", ".join(FEATURES)}.items()],
                  columns=["Setting", "Value"])

for name, t in [("Table1_sample_size", T1), ("Table2_performance", T2), ("Table3_settings", T3)]:
    t.to_csv(os.path.join(OUT_DIR, name + ".csv"), index=False, encoding="utf-8-sig")

for k in SETS:
    s = df.loc[masks[k], ["year", "month", "row", "col"]].copy()
    s["Observed_CDI"], s["Simulated_CDI"] = y_all[masks[k]], preds[k]
    s.to_csv(os.path.join(OUT_DIR, f"predictions_{k}.csv"), index=False)

print("\nTABLE 1: Sample Size Summary\n", T1.to_string(index=False))
print("\nTABLE 2: Performance Summary\n", T2.to_string(index=False))

# =============================================================================
# FIGURE 1: SCATTER PLOTS (Training, Validation, Testing)
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(24, 8))
vmin = min(preds[k].min() for k in SETS)
vmax = max(preds[k].max() for k in SETS)

for i, (ax, k, lt) in enumerate(zip(axes, SETS, "abc")):
    o, p = y_all[masks[k]], preds[k]
    lo, hi = min(o.min(), p.min()), max(o.max(), p.max())
    pad = 0.05 * (hi - lo)
    lo, hi = lo - pad, hi + pad
    
    sc = ax.scatter(o, p, s=12, alpha=0.25, c=p, cmap="RdYlBu_r", vmin=vmin,
                    vmax=vmax, edgecolors="none", rasterized=True)
    ax.plot([lo, hi], [lo, hi], "k--", linewidth=2, label="1:1 line")
    
    s_, i_ = np.polyfit(o, p, 1)
    xf = np.linspace(lo, hi, 200)
    ax.plot(xf, s_ * xf + i_, color="#d7191c", linewidth=2, label=f"Fit (slope = {s_:.2f})")
    
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_aspect("equal", adjustable="box")
    
    # X-axis label
    ax.set_xlabel("Observed CDI", fontsize=21, fontweight="bold")
    
    # Y-axis label only on the very left side
    if i == 0:
        ax.set_ylabel("Simulated (LightGBM) CDI", fontsize=21, fontweight="bold")
    else:
        ax.set_ylabel("")
        
    ax.text(0.5, 1.02, lt, transform=ax.transAxes, fontsize=FS + 2,
            fontweight="bold", ha="center", va="bottom")
    
    mt = met[k]
    ax.text(0.05, 0.95,
            f"{k} (n = {len(o):,})\n$R^2$ = {mt['R2']:.2f}\n"
            f"RMSE = {mt['RMSE']:.2f}\nMAE = {mt['MAE']:.2f}",
            transform=ax.transAxes, fontsize=FS - 0, va="top",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="grey", alpha=0.9))
    ax.legend(loc="lower right", framealpha=0.85)
    ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
cax = fig.add_axes([1.01, 0.20, 0.015, 0.60])
cb = fig.colorbar(sc, cax=cax)
cb.set_label("CDI", fontsize=FS)
fig.savefig(os.path.join(OUT_DIR, "Figure_scatter_train_val_test.png"), dpi=400, bbox_inches="tight")
plt.close(fig)

# =============================================================================
# FIGURE 2: LEARNING CURVE (Training and Validation RMSE)
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(evals["train"]["rmse"], linewidth=2.5, label="Training RMSE")
ax.plot(evals["val"]["rmse"], linewidth=2.5, linestyle="--", label="Validation RMSE")
ax.axvline(x=best_round, color="green", linestyle=":", linewidth=2, label=f"Best Round ({best_round})")
ax.set_xlabel("Boosting round")
ax.set_ylabel("RMSE")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "Figure_learning_curve.png"), dpi=350, bbox_inches="tight")
plt.close(fig)

print("\nAll files successfully saved to:", OUT_DIR)
print("Process completed.")

LightGBM: CHIRPS + ERA5 -> CDI | Train, Validation, and Test Split
CDI months: 311 (2000_02 to 2025_12)
Total samples across all years: 45,406
Training until validation scores don't improve for 50 rounds
[50]	train's rmse: 0.254629	val's rmse: 0.279698
[100]	train's rmse: 0.191613	val's rmse: 0.243564
[150]	train's rmse: 0.176025	val's rmse: 0.242474
[200]	train's rmse: 0.16543	val's rmse: 0.241914
[250]	train's rmse: 0.156801	val's rmse: 0.242108
Early stopping, best iteration is:
[201]	train's rmse: 0.165251	val's rmse: 0.241904

Best boosting round selected by validation set: 201

TABLE 1: Sample Size Summary
    Dataset     Years  Months  Pixels  Samples  Share (%)  CDI mean  CDI SD  CDI min  CDI max
  Training 2000-2017     215     146    31390       69.1     -0.62    1.66    -3.46     3.82
Validation 2018-2021      48     146     7008       15.4     -0.51    1.66    -3.68     3.33
   Testing 2022-2025      48     146     7008       15.4     -0.44    1.64    -2.81     3.50

TABLE 

## LightGBM-future CDI

In [5]:
# =============================================================================
# FUTURE CDI PROJECTION (2030-2050) WITH THE FINAL LightGBM MODEL
# =============================================================================

import os
import glob
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import rasterio
import joblib
import lightgbm as lgb

# =============================================================================
# PATHS AND SETTINGS
# =============================================================================

BASE_DIR   = r"C:\Users\gerii\Documents\projects\tekeze"
NEX_DIR    = os.path.join(BASE_DIR, "NEX-GDDP-CMIP6")
MODEL_DIR  = os.path.join(BASE_DIR, "lgbm_results_final")
CDI_DIR    = os.path.join(BASE_DIR, "indices", "CDI_OUTPUT_025")
CDI_PREFIX = "CDI_"

PR_COR = os.path.join(NEX_DIR, "pr_corrected_RF", "ENSEMBLE")
TA_COR = os.path.join(NEX_DIR, "ta_corrected_RF", "ENSEMBLE")

OUT_DIR = os.path.join(NEX_DIR, "CDI_future_final")
os.makedirs(OUT_DIR, exist_ok=True)

SCENARIOS = ["ssp245", "ssp585"]
PRED_Y0, PRED_Y1 = 2030, 2050
NODATA = -9999.0

# =============================================================================
# LOAD MODEL, SCALER AND FEATURES
# =============================================================================

with open(os.path.join(MODEL_DIR, "final_features.json")) as f:
    cfg = json.load(f)
FEATURES = cfg["features"]
model  = lgb.Booster(model_file=os.path.join(MODEL_DIR, "lgbm_model_final.txt"))
scaler = joblib.load(os.path.join(MODEL_DIR, "scaler_final.joblib"))

print("=" * 70)
print("FUTURE CDI PROJECTION (2030_01 - 2050_12)")
print("=" * 70)

# =============================================================================
# HELPERS
# =============================================================================

def load_band(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        nd = src.nodata
    if nd is not None and not np.isnan(nd):
        arr[arr == nd] = np.nan
    arr[arr <= -9999] = np.nan
    return arr

def load_series(folder, y0, y1, prefix=""):
    d = {}
    for y in range(y0, y1 + 1):
        for m in range(1, 13):
            k = f"{y}_{m:02d}"
            p = os.path.join(folder, f"{prefix}{k}.tif")
            if os.path.exists(p):
                d[k] = load_band(p)
    return d

def back(year, month, k):
    idx = year * 12 + (month - 1) - k
    return f"{idx // 12}_{idx % 12 + 1:02d}"

def features_for(P, T, key):
    y_, m_ = int(key[:4]), int(key[5:7])
    lk = [back(y_, m_, k) for k in range(6)]
    
    # Fallback to the first available month if 2029 lag files don't exist in folder
    fallback_p = next(iter(P.values())) if P else None
    fallback_t = next(iter(T.values())) if T else None
    
    resolved_p = [P.get(k, fallback_p) for k in lk]
    resolved_t = [T.get(k, fallback_t) for k in lk]
    
    if any(p is None for p in resolved_p) or any(t is None for t in resolved_t):
        return None
        
    return {
        "precip": resolved_p[0], "tmean": resolved_t[0],
        "precip_lag1": resolved_p[1], "tmean_lag1": resolved_t[1],
        "precip_3m": np.sum(resolved_p[:3], axis=0),
        "precip_6m": np.sum(resolved_p[:6], axis=0),
        "tmean_3m":  np.mean(resolved_t[:3], axis=0),
    }

# =============================================================================
# REFERENCE MASK AND PROFILE
# =============================================================================

cdi_obs = load_series(CDI_DIR, 2000, 2025, prefix=CDI_PREFIX)
mask = np.zeros(next(iter(cdi_obs.values())).shape, dtype=bool)
for a in cdi_obs.values():
    mask |= np.isfinite(a)

ref = next(iter(glob.glob(os.path.join(PR_COR, "ssp245", "*.tif"))), None)
if ref is None:
    raise SystemExit("No corrected ensemble files found for ssp245.")
with rasterio.open(ref) as r:
    profile = r.profile.copy()
profile.update(dtype="float32", count=1, nodata=NODATA, compress="lzw")
print("Basin pixels:", int(mask.sum()))

# =============================================================================
# RUN PROJECTIONS (OUTPUTS START AT 2030_01)
# =============================================================================

pred_keys = [f"{y}_{m:02d}" for y in range(PRED_Y0, PRED_Y1 + 1) for m in range(1, 13)]

for sc in SCENARIOS:
    sc_dir_pr = os.path.join(PR_COR, sc)
    sc_dir_ta = os.path.join(TA_COR, sc)
    
    P_sc = load_series(sc_dir_pr, PRED_Y0, PRED_Y1)
    T_sc = load_series(sc_dir_ta, PRED_Y0, PRED_Y1)
    
    if not P_sc or not T_sc:
        raise SystemExit(f"Missing corrected ensemble data for {sc}.")

    sc_out_dir = os.path.join(OUT_DIR, sc)
    os.makedirs(sc_out_dir, exist_ok=True)
    
    count = 0
    for k in pred_keys:
        f = features_for(P_sc, T_sc, k)
        if f is None:
            continue
        valid = mask.copy()
        for a in f.values():
            valid &= np.isfinite(a)
        if valid.sum() == 0:
            continue
            
        X = np.column_stack([f[c][valid] for c in FEATURES])
        pred = model.predict(scaler.transform(X))
        
        m2 = np.full(mask.shape, NODATA, np.float32)
        m2[valid] = pred
        
        out_path = os.path.join(sc_out_dir, f"CDI_{sc}_{k}.tif")
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(m2, 1)
        count += 1
        
    print(f">> {sc}: Successfully saved {count} monthly GeoTIFFs (starting from 2030_01).")

print("\nAll future CDI projection TIF files successfully generated in:", OUT_DIR)

FUTURE CDI PROJECTION (2030_01 - 2050_12)
Basin pixels: 165
>> ssp245: Successfully saved 252 monthly GeoTIFFs (starting from 2030_01).
>> ssp585: Successfully saved 252 monthly GeoTIFFs (starting from 2030_01).

All future CDI projection TIF files successfully generated in: C:\Users\gerii\Documents\projects\tekeze\NEX-GDDP-CMIP6\CDI_future_final


# visulization

In [6]:
# =============================================================================
# VISUALIZE MONTHLY CDI PROJECTIONS FOR 2030 WITH DEGREE COORDINATES
# =============================================================================

import os
import rasterio
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

# =============================================================================
# PATHS AND SETTINGS
# =============================================================================

BASE_DIR = r"C:\Users\gerii\Documents\projects\tekeze"
OUT_DIR  = os.path.join(BASE_DIR, "NEX-GDDP-CMIP6", "CDI_future_final")
VIS_DIR  = os.path.join(OUT_DIR, "visualization_2030")
os.makedirs(VIS_DIR, exist_ok=True)

SCENARIOS = ["ssp245", "ssp585"]
YEAR = 2030

# Drought colormap (Red = Extreme Drought, Green = No Drought)
CMAP = "RdYlGn"
VMIN, VMAX = -2.5, 1.0  # Adjust if needed based on your CDI value range

print("=" * 70)
print(f"VISUALIZING MONTHLY CDI MAPS FOR {YEAR} WITH DEGREE COORDS")
print("=" * 70)

for sc in SCENARIOS:
    sc_dir = os.path.join(OUT_DIR, sc)
    sc_vis_dir = os.path.join(VIS_DIR, sc)
    os.makedirs(sc_vis_dir, exist_ok=True)
    
    for m in range(1, 13):
        key = f"{YEAR}_{m:02d}"
        tif_path = os.path.join(sc_dir, f"CDI_{sc}_{key}.tif")
        
        if not os.path.exists(tif_path):
            print(f"Missing: {tif_path}")
            continue
            
        with rasterio.open(tif_path) as src:
            arr = src.read(1)
            nodata = src.nodata
            # Get geographic bounding box (left, right, bottom, top) in degrees
            bounds = src.bounds
            extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
            
        # Mask out nodata for plotting
        if nodata is not None:
            arr = arr.astype(float)
            arr[arr == nodata] = float('nan')
            
        # Plotting the map
        fig, ax = plt.subplots(figsize=(8, 7))
        
        im = ax.imshow(arr, cmap=CMAP, vmin=VMIN, vmax=VMAX, extent=extent, origin="upper")
        
        ax.set_title(f"{sc.upper()} - {YEAR}-{m:02d}", fontsize=14, fontweight="bold", pad=10)
        ax.set_xlabel("Longitude (°E)", fontsize=12)
        ax.set_ylabel("Latitude (°N)", fontsize=12)
        
        # Enable grid lines showing degree coordinates
        ax.grid(True, linestyle="--", alpha=0.5, color="gray")
        
        # Add colorbar
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.15)
        cb = fig.colorbar(im, cax=cax)
        cb.set_label("Composite Drought Index (CDI)", fontsize=11)
        
        plt.tight_layout()
        out_fig = os.path.join(sc_vis_dir, f"CDI_{sc}_{key}.png")
        fig.savefig(out_fig, dpi=300, bbox_inches="tight")
        plt.close(fig)
        
    print(f">> Saved 12 monthly visualization maps for {sc} in: {sc_vis_dir}")

print("\nVisualization completed successfully.")

VISUALIZING MONTHLY CDI MAPS FOR 2030 WITH DEGREE COORDS
>> Saved 12 monthly visualization maps for ssp245 in: C:\Users\gerii\Documents\projects\tekeze\NEX-GDDP-CMIP6\CDI_future_final\visualization_2030\ssp245
>> Saved 12 monthly visualization maps for ssp585 in: C:\Users\gerii\Documents\projects\tekeze\NEX-GDDP-CMIP6\CDI_future_final\visualization_2030\ssp585

Visualization completed successfully.
